# 06c · DOSE at ADM1: step C first stages

Fork `fork/corridors-dose`, Part 2. Rules: [`docs/successors/dose-prespec.md`](../../docs/successors/dose-prespec.md)
§3, §3a, §3b; outcomes recorded in §4. **Loads no temperature.** Same rows as step B
(`06b_dose_step_b.ipynb`); only the growth measure changes.

◇ Step C gate, per instrument: first stage of ln(GRP per capita, 2015 LCU) on the instrument,
with precipitation `P_a`, region + country × year FE, weighted by the region's panel cells,
clustered by region. Continue to LST with that instrument only if **F ≥ 10**.

Known problem (prespec §3): a mine's output counts in its region's GRP, so the mine
instruments' first stages are partly mechanical.

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT))
warnings.filterwarnings("ignore")
from src.analysis import dose as D

s = D.step_sample(D.build_region_panel())
print(f"step sample: {len(s):,} region-years, {s.GID_1.nunique():,} regions, "
      f"{s.GID_0.nunique()} countries, {s.year.min()}-{s.year.max()}")

## ◇ Gate: the three first stages

In [ ]:
gate = pd.DataFrame([dict(name=k, **D.first_stage(s, "log_grp", c)) for k, c in D.INSTRUMENTS.items()])
gate["passes"] = gate.F >= 10
gate.round(5)

## Information (not gated)

In [ ]:
info = [
    ("mine count, ln(1 + count)", D.first_stage(s, "log_grp", "log1p_mine_count")),
    ("mine count, country clusters", D.first_stage(s, "log_grp", "mine_count", vcov=D.CL_CTY)),
    ("mine x price, moving exposure", D.first_stage(s, "log_grp", "price_moving")),
    ("mine x price, country clusters", D.first_stage(s, "log_grp", "price_frozen", vcov=D.CL_CTY)),
    ("favoritism, any day", D.first_stage(s, "log_grp", "fav_any")),
    ("favoritism, lagged one year", D.first_stage(s, "log_grp", "fav_share_lag1")),
    ("favoritism, country clusters", D.first_stage(s, "log_grp", "fav_share", vcov=D.CL_CTY)),
]
pd.DataFrame([dict(check=k, **r) for k, r in info]).drop(columns=["endog"]).round(5)

Mine × price, leaving one commodity out (only 12 price series drive the instrument).

In [ ]:
exp = pd.read_parquet(f"{D.INSTR}/adm1_price_exposure.parquet")
rows = []
for c in sorted(exp.commodity.unique()):
    t = s.assign(price_loo=D.price_leave_one_out(s, c))
    rows.append(dict(left_out=c, **D.first_stage(t, "log_grp", "price_loo")))
pd.DataFrame(rows).drop(columns=["endog", "instrument"]).round(5)